In [ ]:
# ==========================================
# Cell 1: Setup and Libraries
# ==========================================
import cv2
import mediapipe as mp
import os
import csv
import numpy as np

print("✅ Libraries loaded for Keypoint Extraction.")


In [ ]:
# ==========================================
# Cell 2: Project Configuration
# ==========================================
# 1. SET YOUR DATASET DIRECTORY HERE
DATASET_DIR = "dataset"

# 2. Define the custom classes you want to add
custom_classes = ['space', 'delete', 'nothing']
images_to_collect_per_class = 200

# Create the folders if they don't exist
for cls in custom_classes:
    os.makedirs(os.path.join(DATASET_DIR, cls), exist_ok=True)
    
print(f"✅ Dataset directory '{os.path.abspath(DATASET_DIR)}' is ready.")


In [ ]:
# ==========================================
# Cell 3: The Smart Data Collector
# ==========================================
# Initialize MediaPipe Hands
mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils
hands = mp_hands.Hands(static_image_mode=False, max_num_hands=2, min_detection_confidence=0.5)

cap = cv2.VideoCapture(0)

print("📸 Opening webcam... Check your taskbar if the window doesn't pop up immediately.")
print("Follow the on-screen instructions.")

for cls in custom_classes:
    print(f"\n--- GET READY FOR: '{cls}' ---")
    
    # Wait for user to press 's'
    ready = False
    while not ready:
        ret, frame = cap.read()
        if not ret: continue
            
        # Show hand tracking in the "Ready" screen
        image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = hands.process(image_rgb)
        if results.multi_hand_landmarks:
            for hand_landmarks in results.multi_hand_landmarks:
                mp_drawing.draw_landmarks(frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)
                
        cv2.putText(frame, f"Ready for: {cls}? Press 's' to start", (20, 50), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
        cv2.imshow('Smart Collector', frame)
        
        key = cv2.waitKey(1) & 0xFF
        if key == ord('s'):
            ready = True
        elif key == ord('q'):
            cap.release()
            cv2.destroyAllWindows()
            print("Collection aborted by user.")
            exit()
            
    # Record loop
    print(f"Recording {cls}...")
    img_num = 0
    
    while img_num < images_to_collect_per_class:
        ret, frame = cap.read()
        if not ret: continue
            
        clean_frame = frame.copy() # MUST save a clean copy of the frame!
        
        # Process the frame to check for hands
        image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = hands.process(image_rgb)
        hand_detected = results.multi_hand_landmarks is not None
        
        # Smart Recording Logic
        is_valid_frame = False
        if cls == 'nothing' and not hand_detected:
            is_valid_frame = True
        elif cls != 'nothing' and hand_detected:
            is_valid_frame = True
            # Draw landmarks on the DISPLAY frame only
            for hand_landmarks in results.multi_hand_landmarks:
                mp_drawing.draw_landmarks(frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)

        if is_valid_frame:
            img_path = os.path.join(DATASET_DIR, cls, f"{cls}_{int(time.time()*1000)}_{img_num}.jpg")
            cv2.imwrite(img_path, clean_frame)
            img_num += 1
            cv2.putText(frame, f"Recording {cls}: {img_num}/{images_to_collect_per_class}", (20, 50), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
        else:
            msg = "WAITING: No hand detected!" if cls != 'nothing' else "WAITING: Hand in frame!"
            cv2.putText(frame, msg, (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)

        cv2.imshow('Smart Collector', frame)
        if cv2.waitKey(1) & 0xFF == ord('q'): break

print("\n✅ Collection Complete! Webcam released.")
cap.release()
cv2.destroyAllWindows()


In [ ]:
# ==========================================
# Cell 4: Visual Check (Grid Layout) - FIXED
# ==========================================
print(f"Visualizing dataset: {os.path.abspath(DATASET_DIR)}\n")

classes = [d for d in os.listdir(DATASET_DIR) if os.path.isdir(os.path.join(DATASET_DIR, d))]

# Calculate Grid Layout (6 columns wide)
cols = 6 
rows = math.ceil(len(classes) / cols)

# Set up matplotlib grid
fig, axes = plt.subplots(rows, cols, figsize=(18, 3 * rows))

# THE FIX: Since cols is always 6, axes is always a numpy array. We safely flatten it.
axes = axes.flatten()

for idx, cls in enumerate(classes):
    class_path = os.path.join(DATASET_DIR, cls)
    image_files = [f for f in os.listdir(class_path) if f.endswith(('.jpg', '.png', '.jpeg'))]
    
    if len(image_files) > 0:
        random_file = random.choice(image_files)
        sample_path = os.path.join(class_path, random_file)
        sample_img = cv2.imread(sample_path)
        sample_img_rgb = cv2.cvtColor(sample_img, cv2.COLOR_BGR2RGB)
        
        axes[idx].imshow(sample_img_rgb)
        axes[idx].set_title(f"{cls}\n({len(image_files)} imgs)")
        axes[idx].axis('off')
    else:
        axes[idx].set_title(f"{cls}\n(EMPTY)")
        axes[idx].axis('off')

# Hide any extra empty plots
for j in range(len(classes), len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.show()


In [ ]:
# ==========================================
# Cell 5: Strict Machine Learning Validation
# ==========================================
print("🔍 Starting Strict Dataset Validation...")
print("This might take a minute or two depending on dataset size.\n")

mp_hands = mp.solutions.hands
hands_validator = mp_hands.Hands(static_image_mode=True, max_num_hands=2, min_detection_confidence=0.5)

total_images_scanned = 0
bad_images_found = 0

for cls in classes:
    class_path = os.path.join(DATASET_DIR, cls)
    image_files = [f for f in os.listdir(class_path) if f.endswith(('.jpg', '.png', '.jpeg'))]
    class_bad_images = []
    
    for img_name in image_files:
        total_images_scanned += 1
        img_path = os.path.join(class_path, img_name)
        img = cv2.imread(img_path)
        
        if img is None:
            class_bad_images.append((img_name, "File Corrupted / Unreadable"))
            continue
            
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        results = hands_validator.process(img_rgb)
        hand_detected = results.multi_hand_landmarks is not None
        
        if cls != 'nothing' and not hand_detected:
            class_bad_images.append((img_name, "No hand detected"))
        elif cls == 'nothing' and hand_detected:
             class_bad_images.append((img_name, "Accidental hand detected"))

    if len(class_bad_images) > 0:
        print(f"❌ Class '{cls}' has {len(class_bad_images)} problematic images:")
        for bad_img, reason in class_bad_images:
            print(f"   - {bad_img}: {reason}")
            bad_images_found += 1
    else:
        print(f"✅ Class '{cls}' is perfect! ({len(image_files)} valid images)")

print("\n====================================")
print(f"Validation Complete! Scanned {total_images_scanned} images.")
print(f"Total Problematic Images Found: {bad_images_found}")
if bad_images_found > 0:
    print("⚠️ Please delete the flagged images from your dataset folder.")


In [ ]:
# ==========================================
# Cell 6: Package Dataset for the Team
# ==========================================
OUTPUT_FILENAME = "Bilingual_Sign_Language_Dataset"

print(f"📦 Zipping the '{DATASET_DIR}' folder...")
shutil.make_archive(OUTPUT_FILENAME, 'zip', DATASET_DIR)

print(f"✅ Success! Your dataset is packaged as: {OUTPUT_FILENAME}.zip")
print(f"Location: {os.path.abspath(OUTPUT_FILENAME + '.zip')}")
